In [2]:
!pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]widgets]


In [ ]:
"""
Interactive visualizer for your SpeechMasker, for Jupyter / IPython.

It calls YOUR real masking code:
  - speech_jepa.audio_masking.compute_mask_indices
  - SpeechMasker.filter_small_clusters

The sampling loop below mirrors SpeechMasker.forward exactly, with ONE addition:
a `max_attempts` guard. Your forward's `while True:` has no cap, so a ratio_cutoff
that can't be satisfied (given target_prob / target_length / groups) will spin
forever and hang the kernel. Worth adding a cap to the real class too.

Usage in a notebook cell:
    from speech_masker_viz import launch
    launch()                              # auto-resolves your imports
    # or, if auto-resolve can't find the class / wrong path:
    from your.module import SpeechMasker
    launch(masker_cls=SpeechMasker)

Color scheme (matches the inline widget):
    context (encoder sees)  green
    target group g          its own color
    fully masked (orphan)   gray
    padding                 dark slate
Rows:
    Global    true per-frame category
    Target g  one decoder pass (context faint-green, its targets in color, rest gray)
    Encoder   exactly what the student attends to
"""

import importlib
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as W
from IPython.display import display
import sys
sys.path.append('../')

def _resolve():
    try:
        from speech_jepa.audio_masking import compute_mask_indices
    except Exception as e:
        raise ImportError(
            "Could not import compute_mask_indices from speech_jepa.audio_masking. "
            "Edit _resolve() to match your path."
        ) from e

    speech_masker = None
    for p in [
        "speech_jepa.masking",
        "speech_jepa.maskers",
        "speech_jepa.audio_masking",
        "speech_jepa.modules.masking",
        "speech_jepa.data.masking",
        "speech_jepa.modules",
    ]:
        try:
            mod = importlib.import_module(p)
            if hasattr(mod, "SpeechMasker"):
                speech_masker = getattr(mod, "SpeechMasker")
                print(f"[viz] SpeechMasker imported from {p}")
                break
        except Exception:
            continue
    if speech_masker is None:
        raise ImportError(
            "Could not locate SpeechMasker automatically. "
            "Pass it explicitly: launch(masker_cls=YourSpeechMasker)."
        )
    return speech_masker, compute_mask_indices


def _safe_sample(masker, compute_mask_indices, n_times, max_attempts=200):
    """Mirror of SpeechMasker.forward (in_channels=1, channel_based_masking=False),
    with a max_attempts guard added. Returns (context_bool, targets_bool, attempts, ratio)."""
    attempts = 0
    G = masker.target_masks_per_context
    while True:
        attempts += 1
        targets = torch.zeros([G, n_times], dtype=torch.bool)
        for g in range(G):
            t = compute_mask_indices(
                shape=(1, n_times),
                padding_mask=None,
                mask_prob=masker.target_prob,
                mask_length=masker.target_length,
            )
            targets[g] = t.reshape(-1)[:n_times]
        any_target = torch.any(targets, dim=0)
        context = ~any_target
        context = masker.filter_small_clusters(context)          # YOUR method
        ratio = float(torch.sum(context).item()) / n_times
        if ratio >= masker.ratio_cutoff or attempts >= max_attempts:
            break
    return context.bool(), targets.bool(), attempts, ratio


def _hex(h):
    h = h.lstrip("#")
    return np.array([int(h[i:i + 2], 16) / 255.0 for i in (0, 2, 4)])


_CTX = _hex("#22c55e")
_CTXF = _hex("#bbf7d0")
_ANY = _hex("#6366f1")
_ORPH = _hex("#9ca3af")
_OTHER = _hex("#cbd5e1")
_PAD = _hex("#334155")
_GROUP = [_hex(c) for c in
          ["#3b82f6", "#f59e0b", "#8b5cf6", "#ec4899",
           "#14b8a6", "#ef4444", "#84cc16", "#fb7185"]]


def _fill_row(frames, total, pairs, default_rgb, pad_rgb):
    row = np.tile(pad_rgb, (total, 1))
    real = np.tile(default_rgb, (frames, 1))
    for cond, rgb in pairs:
        real[np.asarray(cond)] = rgb
    row[:frames] = real
    return row


def _render(masker, compute_mask_indices, frames, pad, n_groups,
            target_prob, target_length, min_context_len, ratio_cutoff):
    m = masker(
        target_masks_per_context=n_groups,
        target_prob=target_prob,
        target_length=target_length,
        min_context_len=min_context_len,
        ratio_cutoff=ratio_cutoff,
        channel_based_masking=False,
    )
    context, targets, attempts, ratio = _safe_sample(m, compute_mask_indices, frames)
    ctx = context.numpy()
    tg = targets.numpy()
    any_t = tg.any(axis=0)
    orphan = (~ctx) & (~any_t)
    total = frames + pad

    rows, labels = [], []
    rows.append(_fill_row(frames, total, [(ctx, _CTX), (any_t, _ANY)], _ORPH, _PAD))
    labels.append("Global")
    for g in range(n_groups):
        rows.append(_fill_row(frames, total,
                              [(ctx, _CTXF), (tg[g], _GROUP[g % len(_GROUP)])],
                              _OTHER, _PAD))
        labels.append(f"Target {g}")
    rows.append(_fill_row(frames, total, [(ctx, _CTX)], _ORPH, _PAD))
    labels.append("Encoder")
    img = np.stack(rows, axis=0)
    R = img.shape[0]

    fig, ax = plt.subplots(figsize=(11.5, 0.46 * R + 1.4), dpi=110)
    ax.imshow(img, aspect="auto", interpolation="nearest", extent=[0, total, R, 0])
    ax.set_yticks(np.arange(R) + 0.5)
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("feature frames (20 ms each)", fontsize=10)
    ax.set_xlim(0, total)
    for sp in ["top", "right", "left"]:
        ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)

    if pad > 0:
        ax.axvline(frames, color="#334155", lw=1.3, ls="--")
        ax.text(frames + pad / 2, -0.35, "pad", ha="center", va="bottom",
                fontsize=9, color="#475569")

    secax = ax.secondary_xaxis("top", functions=(lambda x: x / 50.0, lambda s: s * 50.0))
    secax.set_xlabel("seconds", fontsize=9)
    secax.tick_params(labelsize=8, length=0)

    pct = lambda v: f"{round(v * 100):d}%"
    cap_note = "  (hit attempt cap)" if attempts >= 200 else ""
    ax.set_title(
        f"frames={frames} ({frames / 50:.1f}s)   context={pct(ratio)} "
        f"(cutoff {ratio_cutoff:.2f})   targets∪={pct(float(any_t.mean()))}   "
        f"fully-masked={pct(float(orphan.mean()))}   "
        f"mean/group={pct(float(tg.mean()))}   tries={attempts}{cap_note}",
        fontsize=10, loc="left", pad=24,
    )

    handles = [mpatches.Patch(color=_CTX, label="context (encoder sees)")]
    for g in range(min(n_groups, len(_GROUP))):
        handles.append(mpatches.Patch(color=_GROUP[g], label=f"target {g}"))
    handles += [
        mpatches.Patch(color=_ANY, label="any target (global row)"),
        mpatches.Patch(color=_ORPH, label="fully masked"),
        mpatches.Patch(color=_PAD, label="padding"),
    ]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.35),
              ncol=4, frameon=False, fontsize=8.5)
    plt.tight_layout()
    plt.show()


def launch(masker_cls=None, compute_mask_indices_fn=None):
    if masker_cls is None or compute_mask_indices_fn is None:
        resolved_cls, resolved_cmi = _resolve()
        masker_cls = masker_cls or resolved_cls
        compute_mask_indices_fn = compute_mask_indices_fn or resolved_cmi

    cu = dict(continuous_update=False)
    fr = W.IntSlider(value=250, min=40, max=1250, step=5, description="frames", **cu)
    pd = W.IntSlider(value=100, min=0, max=500, step=10, description="padding", **cu)
    ng = W.IntSlider(value=4, min=1, max=8, step=1, description="groups", **cu)
    tp = W.FloatSlider(value=0.15, min=0.05, max=0.5, step=0.01,
                       description="target_prob", readout_format=".2f", **cu)
    tl = W.IntSlider(value=10, min=2, max=40, step=1, description="target_len", **cu)
    mc = W.IntSlider(value=5, min=1, max=30, step=1, description="min_ctx", **cu)
    rc = W.FloatSlider(value=0.5, min=0.05, max=0.8, step=0.01,
                       description="ratio_cut", readout_format=".2f", **cu)
    resample = W.Button(description="Resample", icon="refresh")
    out = W.Output()

    def _update(*_):
        with out:
            out.clear_output(wait=True)
            _render(masker_cls, compute_mask_indices_fn,
                    fr.value, pd.value, ng.value, tp.value,
                    tl.value, mc.value, rc.value)

    for w in (fr, pd, ng, tp, tl, mc, rc):
        w.observe(_update, names="value")
    resample.on_click(lambda b: _update())

    controls = W.VBox([
        W.HBox([fr, pd]),
        W.HBox([ng, tl, mc]),
        W.HBox([tp, rc]),
        resample,
    ])
    display(controls, out)
    _update()

launch()

[viz] SpeechMasker imported from speech_jepa.masking


Output()